# Civitas — train Model A (hate speech detector)

XLM-RoBERTa encoder with 4 heads: hate yes/no, target group, severity, and which words are
hateful (rationale). Languages: en, hi, ta, te, ml, kn, Roman Urdu.

## WHAT YOU DO
1. Do **not** unzip anything. Open https://colab.research.google.com → **File → Upload notebook**
   → pick `train_model_a.ipynb`.
2. **Runtime → Change runtime type → T4 GPU** (L4/A100 if you have Pro), then **Runtime → Run all**.
   Allow Google Drive access when asked (that is where progress is saved).
3. When the cell "Get the bundle" shows a **Choose files** button, pick `civitas_model_a.zip`.
   (Or upload `civitas_model_a.zip` to My Drive beforehand; the notebook then finds it itself.)
4. If Colab disconnects: reconnect and **Run all** again (pick the zip again if asked). Training resumes from the last save
   (at most ~500 steps are repeated).
5. When it finishes, download the one file **My Drive → `civitas_runs/civitas_model_a/civitas_model_a_trained.zip`**
   (trained weights + all results) into your Civitas project folder and run
   `python scripts/install_models.py --check`. Do not unzip it by hand; the script puts
   every file where the app expects it.
6. Do **not** tick CONFIRM in the last cell ("RUN ONCE") until Claude says tuning is finished.

Rough time on a free T4: 40–60 min per epoch; 4 epochs with early stopping ≈ 3–4 h,
across one or two sessions.

In [ ]:
# GPU check: picks batch size / precision for this GPU
import subprocess, torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then Run all again."
GPU = torch.cuda.get_device_name(0)
major, _ = torch.cuda.get_device_capability(0)
BATCH, ACCUM = (64, 1) if "A100" in GPU else (32, 1)
print(f"{GPU}: batch {BATCH} x accum {ACCUM},", "bf16" if major >= 8 else "fp16 + GradScaler")

In [ ]:
# Get the bundle (code + data). Works with ANY of these - no folder upload needed:
#   a) civitas_model_a.zip uploaded to My Drive (any name like "civitas_model_a (1).zip" is fine)
#   b) civitas_model_a.zip dragged into Colab's Files panel (left sidebar)
#   c) nothing uploaded yet: a "Choose files" button appears below - pick the zip
# Checkpoints and the final model file are saved to My Drive/civitas_runs/civitas_model_a/
import glob, hashlib, json, os, shutil, sys, zipfile
NAME = "civitas_model_a"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    MYDRIVE = "/content/drive/MyDrive"
except Exception as e:
    MYDRIVE = None
    print("WARNING: Google Drive not mounted - progress is LOST if Colab disconnects.", e)

def find_zip():
    places = ["/content"] + ([MYDRIVE, f"{MYDRIVE}/civitas", f"{MYDRIVE}/*"] if MYDRIVE else [])
    for d in places:
        hits = sorted(glob.glob(f"{d}/{NAME}*.zip"), key=os.path.getmtime, reverse=True)
        if hits:
            return hits[0]
    for d in ([MYDRIVE, f"{MYDRIVE}/civitas", f"{MYDRIVE}/*"] if MYDRIVE else []):
        for c in glob.glob(f"{d}/{NAME}*/code") + glob.glob(f"{d}/{NAME}*/{NAME}/code"):
            return os.path.dirname(c)          # an unzipped folder on Drive also works
    from google.colab import files
    print(f"Pick {NAME}.zip from your computer:")
    up = files.upload()
    z = next((f for f in up if f.startswith(NAME) and f.endswith(".zip")), None)
    assert z, f"That was not {NAME}.zip - run this cell again and pick the right file."
    return f"/content/{z}"

src = find_zip()
BUNDLE = "/content/bundle"
if os.path.isdir(src):
    shutil.copytree(src, f"{BUNDLE}/{NAME}", dirs_exist_ok=True)
else:
    print("unzipping", src)
    zipfile.ZipFile(src).extractall(BUNDLE)
BUNDLE = f"{BUNDLE}/{NAME}"
WORK, DATA = "/content/work", "/content/data"
RUNS = f"{MYDRIVE}/civitas_runs/{NAME}" if MYDRIVE else f"/content/civitas_runs/{NAME}"
shutil.copytree(f"{BUNDLE}/code", WORK, dirs_exist_ok=True)
shutil.copytree(f"{BUNDLE}/data", DATA, dirs_exist_ok=True)
os.makedirs(RUNS, exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

manifest = json.load(open(f"{DATA}/manifest.json"))
for name, digest in manifest["files"].items():
    h = hashlib.sha256(open(f"{DATA}/{name}", "rb").read()).hexdigest()
    assert h == digest, f"{name} is corrupted (sha256 mismatch) - upload the zip again"
print("bundle from:", src, "| saving runs to:", RUNS)
for split, info in manifest["splits"].items():
    print(f"  {split:>6}: {info['rows']:>7} rows  {info.get('language_counts')}")
print("built from commit", manifest["git_commit"], "at", manifest["created_at"])

In [ ]:
# Install what training needs (Colab already has torch)
!pip -q install "transformers>=4.40" "indic-transliteration>=2.3.45" pyyaml pyarrow scikit-learn sentencepiece
import transformers
print("transformers", transformers.__version__)

In [ ]:
# Training config. Every key is a train.py flag (python train.py --help lists them all)
RUN_NAME = "model_a"  #@param {type:"string"}
OUT = f"{RUNS}/{RUN_NAME}"
CONFIG = {
    "data_dir": DATA,
    "output_dir": OUT,
    "encoder_name": "xlm-roberta-base",
    "epochs": 4,
    "batch_size": BATCH,
    "grad_accum": ACCUM,
    "lr": 2e-5,
    "max_length": 128,
    "warmup_ratio": 0.06,
    "evals_per_epoch": 3,
    "patience": 4,
    "save_every": 500,
    # loss
    "hate_gamma": 0.0,
    "severity_gamma": 2.0,
    "w_hate": 1.0, "w_target": 0.5, "w_severity": 0.5, "w_rationale": 0.5,
    # augmentation (train split only)
    "identity_aug": 1,
    "script_augment_p": 0.3,
    "spelling_augment_p": 0.15,
    "lang_weights": f"{DATA}/data_config.yaml",
}
ARGS = " ".join(f"--{k} {v!r}" if isinstance(v, str) else f"--{k} {v}" for k, v in CONFIG.items())
print(ARGS)

In [ ]:
# Train. Safe to re-run after a disconnect: --resume continues from OUT/last.pt
!cd {WORK} && python train.py {ARGS} --resume

## Results (saved to files)
Scores the best checkpoint on **val** (never test), runs the robustness probe, then writes
the single file to download: My Drive → `civitas_runs/civitas_model_a/civitas_model_a_trained.zip`.

In [ ]:
m = json.load(open(f"{OUT}/best_metrics.json"))
keep = ["threshold", "hate_macro_f1", "hate_roc_auc", "severity_macro_f1", "target_macro_f1",
        "rationale_token_f1", "epoch", "global_step"]
print(json.dumps({k: m.get(k) for k in keep}, indent=1))
for s in ("hate_by_language", "hate_by_source"):
    print(s, {k: (round(v["macro_f1"], 3) if v else None) for k, v in m.get(s, {}).items()})
!cd {WORK} && python -c "import pandas as pd; pd.read_parquet('{DATA}/val.parquet').to_csv('/content/val.csv', index=False)"
!cd {WORK} && python -m eval.probe_model --checkpoint {OUT}/best_model.pt --eval_csv /content/val.csv --out {OUT}/robustness.json
!cd {WORK} && python -m results_io --run_dir {OUT} --kind model_a --data_manifest {DATA}/manifest.json --zip {RUNS}/civitas_model_a_trained.zip
print("DOWNLOAD THIS ONE FILE:", f"{RUNS}/civitas_model_a_trained.zip")

---
# RUN ONCE — final report
**Only after Claude confirms tuning is finished.** The test split is the one honest number you
get; changing anything after looking at it makes it optimistic.

In [ ]:
CONFIRM = False  #@param {type:"boolean"}
assert CONFIRM, "Tick CONFIRM only when tuning is finished (see the note above)."
!cd {WORK} && python train.py --eval_only --split test --data_dir {DATA} --output_dir {OUT} --batch_size {BATCH}
!cd {WORK} && python -m eval.eval_hatecheck --checkpoint {OUT}/best_model.pt --out {OUT}/hatecheck.json
!cd {WORK} && python -m results_io --run_dir {OUT} --kind model_a --data_manifest {DATA}/manifest.json --zip {RUNS}/civitas_model_a_trained.zip
print("DOWNLOAD AGAIN (now includes the test report):", f"{RUNS}/civitas_model_a_trained.zip")